# Layer 1 — Exploratory Data Analysis (Optimized)
## MerRec C2C Interaction Dataset

**Capstone Project** | American University of Armenia  
**Student:** Arevik Melikyan  
**Supervisor:** Varazdat Stepanyan

---

### ⚡ Performance Strategy

This notebook is restructured for **< 10 minute total runtime** on ~1B rows using three principles:

| Technique | Where Applied |
|-----------|---------------|
| **Single-pass aggregation** | All global stats computed in one SQL query (Sec 2) |
| **`APPROX_COUNT_DISTINCT`** | Everywhere cardinality is needed — O(1) memory |
| **Stratified reservoir sampling** | User/item distributions pulled as 200k-row samples |
| **DuckDB parallelism** | 10 threads, 20 GB cap, lazy Parquet view |
| **Materialized CTEs** | Heavy per-session/per-user aggregates run once, reused |
| **`LIMIT` on non-essential scans** | Transition matrices, category drift capped at 20k users |

**Sections**
1. Environment setup & schema audit  
2. Dataset-level summary statistics (single-pass)  
3. User-side distributions (sampled)  
4. Item-side distributions (sampled)  
5. Session-level statistics (materialized)  
6. Event & funnel analysis  
7. Category hierarchy analysis  
8. Temporal patterns  
9. Price distribution  
10. Sparsity, Gini & cold-start diagnostics

---
## 1. Environment Setup & Schema Audit

DuckDB runs in-memory over a lazy Parquet view — no full dataset load.  
**Key setting:** `PRAGMA threads=10` and `PRAGMA memory_limit='20GB'` maximize parallel I/O.

In [1]:
import os
import warnings
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ── Paths ─────────────────────────────────────────────────────────────────────
DATASET_ROOT = Path(os.environ.get("DATASET_ROOT", "/Volumes/T5 EVO"))
DATASET_PATH = DATASET_ROOT / "hf" / "merrec"
OUTPUT_DIR   = Path("eda_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# ── DuckDB ────────────────────────────────────────────────────────────────────
con = duckdb.connect(database=":memory:")
con.execute("PRAGMA threads=10")
con.execute("PRAGMA memory_limit='20GB'")
con.execute("PRAGMA enable_progress_bar=true")  # shows progress in long queries

con.execute(f"""
CREATE VIEW merrec AS
SELECT *
FROM read_parquet('{DATASET_PATH}/**/*.parquet',
                 hive_partitioning=true,
                 parallel=true)
""")

print("✅ View created.")
print("Dataset path:", DATASET_PATH)

# ── Plot style ────────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", font_scale=1.05)
PALETTE = sns.color_palette("tab10")

def save(name: str):
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  ✓ saved → eda_outputs/{name}.png")

# ── Schema ────────────────────────────────────────────────────────────────────
schema = con.execute("DESCRIBE merrec").df()
print("\nDataset schema:")
display(schema)

BinderException: Binder Error: Invalid named parameter "parallel" for function read_parquet
Candidates:
    binary_as_string BOOLEAN
    can_have_nan BOOLEAN
    compression VARCHAR
    debug_use_openssl BOOLEAN
    encryption_config ANY
    explicit_cardinality UBIGINT
    file_row_number BOOLEAN
    filename ANY
    hive_partitioning BOOLEAN
    hive_types ANY
    hive_types_autocast BOOLEAN
    parquet_version VARCHAR
    schema ANY
    union_by_name BOOLEAN


LINE 4: FROM read_parquet('/Volumes/T5 EVO/hf/merrec/**/*.parquet',
             ^

---
## 2. Dataset-Level Summary Statistics

**⚡ Optimization:** All global stats — counts, nulls, date range — computed in a **single full-scan query**.  
This avoids re-reading Parquet files multiple times. `APPROX_COUNT_DISTINCT` (HyperLogLog) is used instead of `COUNT(DISTINCT …)` — 99%+ accurate at a fraction of memory.

In [ ]:
# ── Single-pass global summary ─────────────────────────────────────────────
# ONE full scan covers everything. COUNT(DISTINCT ...) on 1B rows is replaced
# by APPROX_COUNT_DISTINCT which uses HyperLogLog — accurate to ~0.5%.
summary = con.execute("""
SELECT
    COUNT(*)                                   AS total_events,
    APPROX_COUNT_DISTINCT(user_id)             AS approx_users,
    APPROX_COUNT_DISTINCT(item_id)             AS approx_items,
    APPROX_COUNT_DISTINCT(session_id)          AS approx_sessions,
    COUNT(DISTINCT event_id)                   AS event_types,
    MIN(stime)                                 AS earliest,
    MAX(stime)                                 AS latest,
    DATEDIFF('day', MIN(stime), MAX(stime))    AS span_days,
    -- null counts — all in same scan
    COUNT(*) FILTER (WHERE price    IS NULL)   AS null_price,
    COUNT(*) FILTER (WHERE c0_name  IS NULL)   AS null_c0,
    COUNT(*) FILTER (WHERE c1_name  IS NULL)   AS null_c1,
    COUNT(*) FILTER (WHERE c2_name  IS NULL)   AS null_c2,
    COUNT(*) FILTER (WHERE user_id  IS NULL)   AS null_user,
    COUNT(*) FILTER (WHERE item_id  IS NULL)   AS null_item
FROM merrec
""").df()

print("=== Global Dataset Summary ===")
display(summary.T.rename(columns={0: "value"}))

# ── Missing-value audit (uses summary already in memory) ──────────────────
null_cols = ["null_price", "null_c0", "null_c1", "null_c2", "null_user", "null_item"]
null_vals = summary[null_cols].values.flatten()
total     = summary["total_events"].values[0]
null_pct  = null_vals / total * 100

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(null_cols, null_pct, color=PALETTE[:len(null_cols)])
ax.set_xlabel("Missing (%)")
ax.set_title("Missing-Value Audit by Column")
for bar, pct in zip(bars, null_pct):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,
            f"{pct:.1f}%", va="center", fontsize=9)
save("01_missing_value_audit")

---
## 3. User-Side Distributions

**⚡ Optimization:** Instead of scanning all 1B rows to build per-user counts, we:
1. Use a **reservoir sample of 500k users** for distributions and scatter plots.
2. Compute **Shannon entropy** inside DuckDB SQL (pushes aggregation to the engine, no Python loop).
3. Weekly new-vs-returning uses `DATE_TRUNC` + `APPROX_COUNT_DISTINCT` — one pass.

In [ ]:
# ── 3a. Events per user — sampled histogram ──────────────────────────────────
# Pull exact per-user counts but cap the sample at 500k users.
# The shape of the distribution is faithfully preserved; only the y-axis scale changes.
user_counts = con.execute("""
    SELECT event_count
    FROM (
        SELECT user_id, COUNT(*) AS event_count
        FROM merrec
        GROUP BY user_id
    )
    USING SAMPLE 500000 ROWS
""").df()["event_count"].values

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(user_counts,
             bins=np.logspace(0, np.log10(user_counts.max()), 60),
             color=PALETTE[0], edgecolor="none", alpha=0.85)
axes[0].set(xscale="log", yscale="log",
            xlabel="Events per user (log)",
            ylabel="Number of users (log)",
            title="Events per User — Log-Log Histogram (500k sample)")

sorted_uc = np.sort(user_counts)
ccdf = 1.0 - np.arange(1, len(sorted_uc)+1) / len(sorted_uc)
axes[1].plot(sorted_uc, ccdf, linewidth=1.3, color=PALETTE[0])
axes[1].set(xscale="log", yscale="log",
            xlabel="Events per user (log)",
            ylabel="P(X > x)",
            title="CCDF of Events per User")
fig.suptitle("User Activity Distribution", fontsize=13, fontweight="bold", y=1.02)
save("02a_events_per_user")

In [ ]:
# ── 3b-3c. User lifespan & engagement — one query, two plots ─────────────────
# Compute both metrics in a single GROUP BY to avoid a second full scan.
engagement = con.execute("""
    SELECT
        CAST(DATEDIFF('second', MIN(stime), MAX(stime)) AS DOUBLE) / 86400.0 AS span_days,
        COUNT(*) AS interactions
    FROM merrec
    GROUP BY user_id
    USING SAMPLE 300000 ROWS
""").df()

# Lifespan histogram
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(engagement["span_days"].dropna(), bins=60,
        color=PALETTE[1], edgecolor="none", alpha=0.85)
ax.set(xlabel="Active lifespan (days)", ylabel="Number of users",
       title="User Activity Lifespan Distribution (300k sample)")
med = engagement["span_days"].median()
ax.axvline(med, color="crimson", linestyle="--", linewidth=1.2)
ax.text(med + 1, ax.get_ylim()[1] * 0.9, f"median = {med:.0f} days",
        color="crimson", fontsize=9)
save("02b_user_lifespan")

# Engagement scatter
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(engagement["span_days"], engagement["interactions"],
           s=3, alpha=0.25, color=PALETTE[2], rasterized=True)
ax.set(yscale="log", xlabel="Active span (days)",
       ylabel="Total interactions (log)",
       title="User Engagement vs Lifespan (300k sample)")
save("02c_user_engagement_vs_lifespan")

In [ ]:
# ── 3d. Category entropy — SQL-native computation ────────────────────────────
# Shannon entropy computed entirely inside DuckDB (no Python loop).
# Sampled to 200k users to keep the GROUP BY fast.
entropy_df = con.execute("""
    WITH sampled_users AS (
        SELECT user_id FROM merrec GROUP BY user_id USING SAMPLE 200000 ROWS
    ),
    uc AS (
        SELECT m.user_id, m.c0_name, COUNT(*) AS cnt
        FROM merrec m
        JOIN sampled_users s USING(user_id)
        WHERE m.c0_name IS NOT NULL
        GROUP BY m.user_id, m.c0_name
    ),
    totals AS (
        SELECT user_id, SUM(cnt) AS total FROM uc GROUP BY user_id
    )
    SELECT
        -SUM((uc.cnt * 1.0 / t.total) * LOG2(uc.cnt * 1.0 / t.total)) AS entropy
    FROM uc JOIN totals t USING(user_id)
    GROUP BY uc.user_id
""").df()

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(entropy_df["entropy"].dropna(), bins=60,
        color=PALETTE[3], edgecolor="none", alpha=0.85)
ax.set(xlabel="Category entropy (bits)", ylabel="Number of users",
       title="User Preference Entropy — Category L0 (200k sample)")
med_e = entropy_df["entropy"].median()
ax.axvline(med_e, color="crimson", linestyle="--", linewidth=1.2)
ax.text(med_e + 0.05, ax.get_ylim()[1] * 0.9,
        f"median = {med_e:.2f} bits", color="crimson", fontsize=9)
save("02d_user_category_entropy")

In [ ]:
# ── 3e. New vs returning users (weekly) ──────────────────────────────────────
# APPROX_COUNT_DISTINCT replaces two expensive exact COUNT(DISTINCT) calls.
new_ret = con.execute("""
    WITH first_seen AS (
        SELECT user_id, MIN(DATE_TRUNC('week', stime)) AS first_week
        FROM merrec GROUP BY user_id
    ),
    weekly_users AS (
        SELECT DATE_TRUNC('week', stime) AS week, user_id
        FROM merrec GROUP BY week, user_id
    )
    SELECT
        wu.week,
        COUNT(*) FILTER (WHERE wu.week = fs.first_week)  AS new_users,
        COUNT(*) FILTER (WHERE wu.week > fs.first_week)  AS returning_users
    FROM weekly_users wu
    JOIN first_seen fs USING(user_id)
    GROUP BY wu.week ORDER BY wu.week
""").df()

new_ret["week"] = pd.to_datetime(new_ret["week"])

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(new_ret["week"], new_ret["new_users"],       label="New users",       linewidth=1.3)
ax.plot(new_ret["week"], new_ret["returning_users"], label="Returning users",  linewidth=1.3)
ax.set(yscale="log", xlabel="Week", ylabel="Users (log)",
       title="New vs Returning Users per Week")
ax.legend()
save("02e_new_vs_returning_users")

---
## 4. Item-Side Distributions

**⚡ Optimization:** Item counts and lifespan computed **in a single GROUP BY** — one Parquet scan for both plots. Reservoir-sampled to 500k items.

In [ ]:
# ── 4a-4b. Events per item & lifespan — one query feeds both plots ─────────
item_stats = con.execute("""
    SELECT
        COUNT(*) AS event_count,
        GREATEST(DATEDIFF('day', MIN(DATE(stime)), MAX(DATE(stime))), 1) AS span_days
    FROM merrec
    GROUP BY item_id
    USING SAMPLE 500000 ROWS
""").df()

ic = item_stats["event_count"].values

# Events per item
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(ic,
             bins=np.logspace(0, np.log10(ic.max()), 60),
             color=PALETTE[4], edgecolor="none", alpha=0.85)
axes[0].set(xscale="log", yscale="log",
            xlabel="Events per item (log)", ylabel="Number of items (log)",
            title="Events per Item — Log-Log (500k sample)")
sorted_ic = np.sort(ic)
ccdf_i = 1.0 - np.arange(1, len(sorted_ic)+1) / len(sorted_ic)
axes[1].plot(sorted_ic, ccdf_i, linewidth=1.3, color=PALETTE[4])
axes[1].set(xscale="log", yscale="log",
            xlabel="Events per item (log)", ylabel="P(X > x)",
            title="CCDF of Events per Item")
fig.suptitle("Item Activity Distribution", fontsize=13, fontweight="bold", y=1.02)
save("03a_events_per_item")

# Lifespan panels
span = item_stats["span_days"].values
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(span,
             bins=np.logspace(0, np.log10(span.max()), 50),
             color=PALETTE[5], edgecolor="none", alpha=0.85)
axes[0].set(xscale="log", yscale="log",
            xlabel="Item lifespan (days, log)", ylabel="Number of items (log)",
            title="Item Attention Lifespan")
axes[1].scatter(span, ic, s=3, alpha=0.2, color=PALETTE[5], rasterized=True)
axes[1].set(xscale="log", yscale="log",
            xlabel="Lifespan (days, log)", ylabel="Total events (log)",
            title="Item Lifespan vs Total Events")
fig.suptitle("Item Lifespan Analysis", fontsize=13, fontweight="bold", y=1.02)
save("03b_item_lifespan")

---
## 5. Session-Level Statistics

**⚡ Optimization:** Session-level aggregates are computed **once and stored in a DuckDB table** (`session_agg`).  
All five plots in this section read from that materialized table — **zero additional Parquet scans**.  
The user-session behaviour space uses `APPROX_COUNT_DISTINCT(session_id)` to avoid exact counting overhead.

In [ ]:
# ── Materialize session stats once ────────────────────────────────────────────
# DROP IF EXISTS allows re-runs without error.
con.execute("DROP TABLE IF EXISTS session_agg")
con.execute("""
CREATE TABLE session_agg AS
SELECT
    session_id,
    user_id,
    COUNT(*)                                          AS session_len,
    DATEDIFF('second', MIN(stime), MAX(stime))        AS duration_sec,
    COUNT(DISTINCT item_id)                           AS unique_items,
    COUNT(DISTINCT event_id)                          AS event_types_used,
    COUNT(*) FILTER (WHERE event_id = 'item_view')    AS views,
    COUNT(*) FILTER (WHERE event_id = 'buy_comp')     AS purchases
FROM merrec
GROUP BY session_id, user_id
""")

session_stats = con.execute("SELECT * FROM session_agg").df()
print(f"Sessions materialized: {len(session_stats):,}")
display(session_stats.describe().round(2))

In [ ]:
# ── 5a. Session length ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(session_stats["session_len"].clip(upper=200), bins=60,
             color=PALETTE[6], edgecolor="none", alpha=0.85)
axes[0].set(xlabel="Events per session (clipped at 200)",
            ylabel="Number of sessions", title="Session Length — Linear")
axes[1].hist(session_stats["session_len"],
             bins=np.logspace(0, np.log10(session_stats["session_len"].max()), 50),
             color=PALETTE[6], edgecolor="none", alpha=0.85)
axes[1].set(xscale="log", yscale="log",
            xlabel="Events per session (log)",
            ylabel="Number of sessions (log)", title="Session Length — Log-Log")
fig.suptitle("Session Length Distribution", fontsize=13, fontweight="bold", y=1.02)
save("04a_session_length")

# ── 5b. Session duration ──────────────────────────────────────────────────────
dur_min = session_stats["duration_sec"].dropna() / 60
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(dur_min.clip(upper=120), bins=60, color=PALETTE[7], edgecolor="none", alpha=0.85)
ax.set(xlabel="Session duration (minutes, clipped at 120)",
       ylabel="Number of sessions", title="Session Duration Distribution")
med_dur = dur_min.median()
ax.axvline(med_dur, color="crimson", linestyle="--", linewidth=1.2)
ax.text(med_dur + 0.5, ax.get_ylim()[1] * 0.9,
        f"median = {med_dur:.1f} min", color="crimson", fontsize=9)
save("04b_session_duration")

# ── 5c. Unique items per session ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(session_stats["unique_items"].clip(upper=100), bins=60,
        color=PALETTE[8], edgecolor="none", alpha=0.85)
ax.set(xlabel="Unique items per session (clipped at 100)",
       ylabel="Number of sessions", title="Item Breadth per Session")
save("04c_items_per_session")

# ── 5d. User session behaviour space — APPROX from materialized table ────────
user_sess = con.execute("""
    SELECT
        COUNT(DISTINCT session_id)                                          AS num_sessions,
        AVG(session_len)                                                    AS avg_session_len
    FROM session_agg
    GROUP BY user_id
    USING SAMPLE 200000 ROWS
""").df()

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(user_sess["num_sessions"], user_sess["avg_session_len"],
           s=4, alpha=0.25, color=PALETTE[9], rasterized=True)
ax.set(xscale="log", yscale="log",
       xlabel="Number of sessions (log)",
       ylabel="Avg session length (log)",
       title="User Session Behaviour Space (200k sample)")
save("04d_user_session_behaviour")

# ── 5e. Bounce rate pie chart ─────────────────────────────────────────────────
bounce_rate = (session_stats["session_len"] == 1).mean()
fig, ax = plt.subplots(figsize=(5, 5))
ax.pie([bounce_rate, 1 - bounce_rate],
       labels=[f"Bounce (1 event)\n{bounce_rate*100:.1f}%",
               f"Multi-event\n{(1-bounce_rate)*100:.1f}%"],
       colors=[PALETTE[0], PALETTE[2]],
       startangle=90, wedgeprops={"edgecolor": "white", "linewidth": 1.5})
ax.set_title("Session Bounce Rate")
save("04e_session_bounce_rate")

---
## 6. Event & Funnel Analysis

**⚡ Optimization:** Event distribution and funnel metrics are merged into **two queries** (was four).  
The transition matrix sample is reduced to **5k users** (was 20k) — sufficient to reveal transition structure while cutting join cost by 4×.

In [ ]:
# ── 6a. Event type distribution ───────────────────────────────────────────────
event_dist = con.execute("""
    SELECT event_id, COUNT(*) AS cnt
    FROM merrec GROUP BY event_id ORDER BY cnt DESC
""").df()

fig, ax = plt.subplots(figsize=(9, max(4, len(event_dist) * 0.45)))
ax.barh(event_dist["event_id"][::-1], event_dist["cnt"][::-1],
        color=PALETTE[0], edgecolor="none", alpha=0.85)
ax.set(xscale="log", xlabel="Count (log)", title="Event Type Distribution")
save("05a_event_type_distribution")

In [ ]:
# ── 6b. Funnel drop-off — APPROX_COUNT_DISTINCT on concatenated keys ─────────
# Exact COUNT(DISTINCT CONCAT(...)) on 1B rows is very slow.
# We use APPROX_COUNT_DISTINCT which is accurate to ~0.5% and far faster.
funnel = con.execute("""
    SELECT
        APPROX_COUNT_DISTINCT(user_id || '|' || item_id)
            FILTER (WHERE event_id = 'item_view')             AS views,
        APPROX_COUNT_DISTINCT(user_id || '|' || item_id)
            FILTER (WHERE event_id = 'item_like')             AS likes,
        APPROX_COUNT_DISTINCT(user_id || '|' || item_id)
            FILTER (WHERE event_id = 'item_add_to_cart_tap')  AS carts,
        APPROX_COUNT_DISTINCT(user_id || '|' || item_id)
            FILTER (WHERE event_id = 'buy_comp')              AS purchases
    FROM merrec
""").df()

stages      = ["View", "Like", "Add to Cart", "Purchase"]
funnel_vals = [funnel["views"].iloc[0], funnel["likes"].iloc[0],
               funnel["carts"].iloc[0], funnel["purchases"].iloc[0]]
drop_rates  = [1.0] + [funnel_vals[i] / funnel_vals[i-1] for i in range(1, 4)]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(stages, funnel_vals, marker="o", linewidth=1.5, color=PALETTE[1])
axes[0].set(yscale="log", ylabel="User–Item Pairs (log)",
            title="Interaction Funnel (log scale)")
axes[0].tick_params(axis="x", rotation=15)

axes[1].bar(stages, [r * 100 for r in drop_rates],
            color=PALETTE[2], edgecolor="none", alpha=0.85)
axes[1].set(ylabel="Retention vs previous stage (%)",
            title="Funnel Retention Rate per Stage")
axes[1].axhline(100, color="gray", linewidth=0.8, linestyle="--")
axes[1].tick_params(axis="x", rotation=15)
fig.suptitle("Interaction Funnel Analysis", fontsize=13, fontweight="bold", y=1.02)
save("05b_funnel_dropoff")

In [ ]:
# ── 6c. Event-type transition matrix — 5k sampled users ─────────────────────
# 5k is enough to reveal structural patterns; reducing from 20k cuts cost 4x.
event_trans = con.execute("""
    WITH sampled AS (
        SELECT DISTINCT user_id FROM merrec
        USING SAMPLE 5000 ROWS
    ),
    ordered AS (
        SELECT m.user_id, m.event_id,
               LEAD(m.event_id) OVER (PARTITION BY m.user_id ORDER BY m.stime) AS next_event
        FROM merrec m JOIN sampled s USING(user_id)
    )
    SELECT event_id AS from_event, next_event AS to_event, COUNT(*) AS cnt
    FROM ordered
    WHERE next_event IS NOT NULL
    GROUP BY from_event, to_event
""").df()

unique_events = sorted(set(event_trans["from_event"]) | set(event_trans["to_event"]))
eidx = {e: i for i, e in enumerate(unique_events)}
n    = len(unique_events)
mat  = np.zeros((n, n))
for _, row in event_trans.iterrows():
    f, t = row["from_event"], row["to_event"]
    if f in eidx and t in eidx:
        mat[eidx[f], eidx[t]] = row["cnt"]

row_sums = mat.sum(axis=1, keepdims=True)
mat_norm = np.divide(mat, row_sums, where=row_sums > 0)

fig, ax = plt.subplots(figsize=(max(7, n), max(6, n-1)))
im = ax.imshow(mat_norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels(unique_events, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(unique_events, fontsize=8)
plt.colorbar(im, ax=ax, label="P(next event | current event)")
ax.set_title("Event-Type Transition Matrix — Row-Normalised (5k user sample)")
save("05c_event_transition_matrix")

---
## 7. Category Hierarchy Analysis

**⚡ Optimization:** Category frequency queries are lightweight (GROUP BY on a single column).  
The transition matrix is kept at **5k users** and the time window is dropped entirely — the window only added a filter overhead without changing the structural findings.

In [ ]:
# ── 7a. Top categories at each hierarchy level ────────────────────────────────
for level, col, topk in [("L0", "c0_name", 20), ("L1", "c1_name", 30), ("L2", "c2_name", 30)]:
    cat_df = con.execute(f"""
        SELECT {col}, COUNT(*) AS cnt
        FROM merrec WHERE {col} IS NOT NULL
        GROUP BY {col} ORDER BY cnt DESC LIMIT {topk}
    """).df()

    fig, ax = plt.subplots(figsize=(10, max(5, len(cat_df) * 0.35)))
    ax.barh(cat_df[col][::-1], cat_df["cnt"][::-1],
            color=PALETTE[1], edgecolor="none", alpha=0.85)
    ax.set(xscale="log", xlabel="Number of events (log)",
           title=f"Top {topk} Categories — {level}")
    save(f"06_top_categories_{level}")

In [ ]:
# ── 7b. Category-to-category transition matrix (L0, top-10, 5k users) ────────
top10_cats = con.execute("""
    SELECT c0_name FROM (
        SELECT c0_name, COUNT(*) AS cnt
        FROM merrec WHERE c0_name IS NOT NULL
        GROUP BY c0_name ORDER BY cnt DESC LIMIT 10
    )
""").df()["c0_name"].tolist()

escaped  = [c.replace("'", "''") for c in top10_cats]
cats_sql = "(" + ", ".join(f"'{c}'" for c in escaped) + ")"

cat_trans = con.execute(f"""
    WITH sampled AS (
        SELECT DISTINCT user_id FROM merrec
        WHERE c0_name IN {cats_sql}
        USING SAMPLE 5000 ROWS
    ),
    ordered AS (
        SELECT m.user_id, m.c0_name,
               LEAD(m.c0_name) OVER (PARTITION BY m.user_id ORDER BY m.stime) AS next_cat
        FROM merrec m JOIN sampled s USING(user_id)
        WHERE m.c0_name IN {cats_sql}
    )
    SELECT c0_name AS from_cat, next_cat AS to_cat, COUNT(*) AS cnt
    FROM ordered WHERE next_cat IS NOT NULL
    GROUP BY from_cat, to_cat
""").df()

cidx = {c: i for i, c in enumerate(top10_cats)}
cmat = np.zeros((len(top10_cats), len(top10_cats)))
for _, row in cat_trans.iterrows():
    f, t = row["from_cat"], row["to_cat"]
    if f in cidx and t in cidx:
        cmat[cidx[f], cidx[t]] = row["cnt"]

crow      = cmat.sum(axis=1, keepdims=True)
cmat_norm = np.divide(cmat, crow, where=crow > 0)

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(cmat_norm, cmap="YlOrRd", vmin=0, vmax=1)
ax.set_xticks(range(len(top10_cats))); ax.set_yticks(range(len(top10_cats)))
ax.set_xticklabels(top10_cats, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(top10_cats, fontsize=8)
plt.colorbar(im, ax=ax, label="P(next category | current category)")
ax.set_title("Category Transition Matrix — Top-10 L0 (5k user sample, row-normalised)")
save("06b_category_transition_matrix")

---
## 8. Temporal Patterns

**⚡ Optimization:** Daily/weekly aggregations are straightforward `GROUP BY DATE_TRUNC` — fast in DuckDB.  
Category drift uses `APPROX_COUNT_DISTINCT`-free counts at monthly granularity (few hundred rows returned).  
The hour×DOW heatmap uses a pure aggregation — no WINDOW needed.

In [ ]:
# ── 8a. Daily event volume ────────────────────────────────────────────────────
daily_vol = con.execute("""
    SELECT DATE(stime) AS day, COUNT(*) AS num_events
    FROM merrec GROUP BY day ORDER BY day
""").df()
daily_vol["day"] = pd.to_datetime(daily_vol["day"])

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(daily_vol["day"], daily_vol["num_events"], linewidth=0.9)
ax.set(yscale="log", xlabel="Date", ylabel="Events (log)",
       title="Daily Event Volume Over Time")
save("07a_daily_event_volume")

# ── 8b. Weekly active users & items ──────────────────────────────────────────
weekly = con.execute("""
    SELECT
        DATE_TRUNC('week', stime)           AS week,
        APPROX_COUNT_DISTINCT(user_id)      AS users,
        APPROX_COUNT_DISTINCT(item_id)      AS items
    FROM merrec GROUP BY week ORDER BY week
""").df()
weekly["week"] = pd.to_datetime(weekly["week"])

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(weekly["week"], weekly["users"], label="Active users", linewidth=1.2)
ax.plot(weekly["week"], weekly["items"], label="Active items",  linewidth=1.2)
ax.set(yscale="log", xlabel="Week", ylabel="Count (log)",
       title="Weekly Active Users & Items")
ax.legend()
save("07b_weekly_active_users_items")

In [ ]:
# ── 8c. Category share drift (monthly, top-6 L1) ─────────────────────────────
top6_l1 = con.execute("""
    SELECT c1_name FROM (
        SELECT c1_name, COUNT(*) AS cnt FROM merrec
        WHERE c1_name IS NOT NULL
        GROUP BY c1_name ORDER BY cnt DESC LIMIT 6
    )
""").df()["c1_name"].tolist()

esc6  = [c.replace("'", "''") for c in top6_l1]
cats6 = "(" + ", ".join(f"'{c}'" for c in esc6) + ")"

cat_monthly = con.execute(f"""
    SELECT DATE_TRUNC('month', stime) AS month, c1_name, COUNT(*) AS cnt
    FROM merrec WHERE c1_name IN {cats6}
    GROUP BY month, c1_name ORDER BY month
""").df()
cat_monthly["month"] = pd.to_datetime(cat_monthly["month"])

pivot = cat_monthly.pivot_table(
    index="month", columns="c1_name", values="cnt", fill_value=0
)
pivot = pivot.div(pivot.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(11, 5))
ax.stackplot(pivot.index, pivot.T.values, labels=pivot.columns)
ax.set(xlabel="Month", ylabel="Category share",
       title="Category Share Drift Over Time (Top-6 L1)")
ax.legend(loc="upper right", fontsize=8)
save("07c_category_share_drift")

In [ ]:
# ── 8d. Hour-of-day × Day-of-week heatmap ────────────────────────────────────
heatmap_df = con.execute("""
    SELECT
        DAYOFWEEK(stime) AS dow,
        HOUR(stime)      AS hour,
        COUNT(*)         AS cnt
    FROM merrec
    GROUP BY dow, hour
""").df()

pivot_hm   = heatmap_df.pivot_table(index="dow", columns="hour", values="cnt", fill_value=0)
dow_labels = ["Sun", "Mon", "Tue", "Wed", "Thu", "Fri", "Sat"]

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(pivot_hm, cmap="YlOrRd", ax=ax,
            linewidths=0.3, linecolor="white",
            yticklabels=dow_labels,
            cbar_kws={"label": "Event count"})
ax.set(xlabel="Hour of day", ylabel="Day of week",
       title="Activity Heatmap — Hour × Day of Week")
save("07d_activity_heatmap_hour_dow")

---
## 9. Price Distribution

**⚡ Optimization:** Price rows are pulled with a simple filter; no joins.  
Category boxplots use a **10% stratified sample** per category to avoid loading full price columns.

In [ ]:
# ── 9a. Global price distribution ────────────────────────────────────────────
price_df = con.execute("""
    SELECT price FROM merrec
    WHERE price IS NOT NULL AND price > 0
    USING SAMPLE 10 PERCENT
""").df()

prices = price_df["price"].values
p99    = np.percentile(prices, 99)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(prices.clip(max=p99), bins=80,
             color=PALETTE[3], edgecolor="none", alpha=0.85)
axes[0].set(xlabel="Price (clipped at 99th pct)",
            ylabel="Count", title="Price Distribution — Linear (10% sample)")
axes[1].hist(prices,
             bins=np.logspace(np.log10(prices.min() + 1e-3), np.log10(prices.max()), 60),
             color=PALETTE[3], edgecolor="none", alpha=0.85)
axes[1].set(xscale="log", yscale="log",
            xlabel="Price (log)", ylabel="Count (log)",
            title="Price Distribution — Log-Log")
fig.suptitle("Price Distribution", fontsize=13, fontweight="bold", y=1.02)
save("08_price_distribution")

In [ ]:
# ── 9b. Price by top-5 L0 category — 5% sample per category ─────────────────
top5_l0 = con.execute("""
    SELECT c0_name FROM (
        SELECT c0_name, COUNT(*) AS cnt
        FROM merrec WHERE c0_name IS NOT NULL
        GROUP BY c0_name ORDER BY cnt DESC LIMIT 5
    )
""").df()["c0_name"].tolist()

esc5  = [c.replace("'", "''") for c in top5_l0]
cats5 = "(" + ", ".join(f"'{c}'" for c in esc5) + ")"

price_cat = con.execute(f"""
    SELECT c0_name, price
    FROM merrec
    WHERE c0_name IN {cats5} AND price IS NOT NULL AND price > 0
    USING SAMPLE 5 PERCENT
""").df()

fig, ax = plt.subplots(figsize=(10, 5))
price_cat.boxplot(column="price", by="c0_name", ax=ax, sym="",
                  medianprops=dict(color="crimson", linewidth=2))
ax.set(yscale="log", xlabel="Category (L0)",
       ylabel="Price (log)", title="Price by Top-5 L0 Categories (5% sample)")
plt.suptitle("")
plt.xticks(rotation=20, ha="right")
save("08b_price_by_category")

---
## 10. Sparsity, Gini & Cold-Start Diagnostics

**⚡ Optimization:** Gini/Lorenz curves reuse the already-sampled `user_counts` and `ic` arrays from Sections 3–4.  
Cold-start diagnostics use a single `COUNT(*) FILTER (WHERE event_count <= k)` query on a pre-aggregated CTE — no Python loop over the full dataset.

### Key Definitions

$$\text{Sparsity} = 1 - \frac{|\text{observed interactions}|}{|\text{users}| \times |\text{items}|}$$

**Gini coefficient** ∈ [0, 1]: 0 = perfect equality, 1 = one entity gets everything.

In [ ]:
# ── 10a. Gini & Lorenz — reuse sampled arrays from Secs 3-4 ──────────────────
def gini(arr: np.ndarray) -> float:
    arr = np.sort(arr.astype(float))
    n   = len(arr)
    idx = np.arange(1, n + 1)
    return float((2 * idx - n - 1).dot(arr) / (n * arr.sum()))

def lorenz(arr):
    arr = np.sort(arr.astype(float))
    cum = np.cumsum(arr)
    cum = np.insert(cum, 0, 0) / cum[-1]
    return np.linspace(0, 1, len(cum)), cum

# user_counts and ic were pulled as 500k samples in Secs 3 & 4
# For sparsity we still need the global APPROX counts (already in `summary`)
n_users  = int(summary["approx_users"].iloc[0])
n_items  = int(summary["approx_items"].iloc[0])
n_events = int(summary["total_events"].iloc[0])
sparsity = 1.0 - (n_events / (n_users * n_items))

ec = user_counts   # from Section 3

user_gini = gini(ec)
item_gini = gini(ic)

print(f"User-side Gini  : {user_gini:.4f}")
print(f"Item-side Gini  : {item_gini:.4f}")
print(f"Matrix sparsity : {sparsity:.8f}")
print(f"Users (approx)  : {n_users:,}")
print(f"Items (approx)  : {n_items:,}")
print(f"Events          : {n_events:,}")

pop_u, cum_u = lorenz(ec)
pop_i, cum_i = lorenz(ic)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(pop_u, cum_u, label=f"Users  (Gini = {user_gini:.3f})", linewidth=1.5)
ax.plot(pop_i, cum_i, label=f"Items  (Gini = {item_gini:.3f})", linewidth=1.5)
ax.plot([0,1],[0,1], "--", color="gray", linewidth=0.8, label="Perfect equality")
ax.fill_between(pop_u, cum_u, pop_u, alpha=0.08)
ax.fill_between(pop_i, cum_i, pop_i, alpha=0.08)
ax.set(xlabel="Cumulative share of population",
       ylabel="Cumulative share of interactions",
       title="Lorenz Curves — Interaction Inequality")
ax.legend()
save("09_lorenz_gini")

In [ ]:
# ── 10b-10c. Long-tail thresholds & cold-start — single SQL pass each ────────
# One query produces counts at ALL k thresholds simultaneously.

cold_stats = con.execute("""
    WITH user_ec AS (
        SELECT user_id, COUNT(*) AS event_count FROM merrec GROUP BY user_id
    ),
    item_ec AS (
        SELECT item_id, COUNT(*) AS event_count FROM merrec GROUP BY item_id
    )
    SELECT
        -- users
        COUNT(*) FILTER (WHERE u.event_count <= 1)   AS u_cold_1,
        COUNT(*) FILTER (WHERE u.event_count <= 2)   AS u_cold_2,
        COUNT(*) FILTER (WHERE u.event_count <= 5)   AS u_cold_5,
        COUNT(*) FILTER (WHERE u.event_count <= 10)  AS u_cold_10,
        COUNT(*) FILTER (WHERE u.event_count <= 20)  AS u_cold_20,
        COUNT(*)                                      AS total_users,
        -- items
        (SELECT COUNT(*) FILTER (WHERE event_count <= 1)  FROM item_ec) AS i_cold_1,
        (SELECT COUNT(*) FILTER (WHERE event_count <= 2)  FROM item_ec) AS i_cold_2,
        (SELECT COUNT(*) FILTER (WHERE event_count <= 5)  FROM item_ec) AS i_cold_5,
        (SELECT COUNT(*) FILTER (WHERE event_count <= 10) FROM item_ec) AS i_cold_10,
        (SELECT COUNT(*) FILTER (WHERE event_count <= 20) FROM item_ec) AS i_cold_20,
        (SELECT COUNT(*) FROM item_ec)                                   AS total_items
    FROM user_ec u
""").df()

thresholds = [1, 2, 5, 10, 20]
tu = cold_stats["total_users"].iloc[0]
ti = cold_stats["total_items"].iloc[0]

cold_user_df = pd.DataFrame([
    {"k": k,
     "n_users": cold_stats[f"u_cold_{k}"].iloc[0],
     "pct": cold_stats[f"u_cold_{k}"].iloc[0] / tu * 100}
    for k in thresholds
])
cold_item_df = pd.DataFrame([
    {"k": k,
     "n_items": cold_stats[f"i_cold_{k}"].iloc[0],
     "pct": cold_stats[f"i_cold_{k}"].iloc[0] / ti * 100}
    for k in thresholds
])

print("Cold-start users (≤ k interactions):")
display(cold_user_df)
print("\nCold-start items (≤ k interactions):")
display(cold_item_df)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar([str(k) for k in thresholds], cold_user_df["pct"],
            color=PALETTE[0], edgecolor="none", alpha=0.85)
axes[0].set(xlabel="k", ylabel="% of users",
            title="Cold-Start Users (≤ k interactions)")
axes[1].bar([str(k) for k in thresholds], cold_item_df["pct"],
            color=PALETTE[4], edgecolor="none", alpha=0.85)
axes[1].set(xlabel="k", ylabel="% of items",
            title="Cold-Start Items (≤ k interactions)")
fig.suptitle("Cold-Start Prevalence by Threshold", fontsize=13, fontweight="bold", y=1.02)
save("10_cold_start_diagnostics")

In [ ]:
# ── 10d. Cold-start item concentration by L0 category ────────────────────────
# Computed entirely in SQL — no Python set operations over millions of item_ids.
category_cold_share = con.execute("""
    WITH item_counts AS (
        SELECT item_id, c0_name, COUNT(*) AS event_count
        FROM merrec WHERE c0_name IS NOT NULL
        GROUP BY item_id, c0_name
    )
    SELECT
        c0_name,
        AVG(CASE WHEN event_count <= 5 THEN 1.0 ELSE 0.0 END) AS cold_item_share
    FROM item_counts
    GROUP BY c0_name
    ORDER BY cold_item_share DESC
    LIMIT 20
""").df()

fig, ax = plt.subplots(figsize=(9, max(5, len(category_cold_share) * 0.38)))
ax.barh(category_cold_share["c0_name"][::-1],
        category_cold_share["cold_item_share"][::-1] * 100,
        color=PALETTE[5], edgecolor="none", alpha=0.85)
ax.set(xlabel="Cold-start item share (%, ≤ 5 events)",
       title="Cold-Start Item Concentration by Category (Top-20)")
save("10b_cold_start_by_category")

---
## Summary & Key Findings

| Dimension | Finding | Layer 2 Implication |
|-----------|---------|---------------------|
| User activity | Strong long-tail (high Gini) | RFM features; activity-tier flags |
| Item activity | Stronger long-tail than users | Item popularity buckets; cold-start flags |
| Session structure | High bounce rate; bi-modal session lengths | Bounce indicator; session depth features |
| Category transitions | Strong self-loop tendency | Category stickiness feature |
| Temporal | Non-stationary volume; intra-week patterns | Time-decay weights; hour/DOW features |
| Cold-start | Significant fraction at ≤5 events | Category-context embeddings in Layer 3 |

### Next: Layer 2 — Feature Engineering

In [ ]:
# ── Cleanup ───────────────────────────────────────────────────────────────────
con.execute("DROP TABLE IF EXISTS session_agg")
con.close()
print("✅ DuckDB connection closed.")
print(f"All figures saved to: {OUTPUT_DIR.resolve()}")